In [6]:
# TÍTULO: Entrenamiento Comparativo - YOLOv8 vs Custom CNN (OCR)
# OBJETIVO: Entrenar y comparar dos arquitecturas para clasificación de caracteres.

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from ultralytics import YOLO
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report
from datetime import datetime
from tqdm.notebook import tqdm

# --- CONFIGURACIÓN DE HARDWARE ---
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f"GPU Activa: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device('cpu')
    print("ALERTA: Usando CPU. El entrenamiento será lento.")

# --- RUTAS ---
# Dataset generado en el paso anterior
DATASET_DIR = '../../datasets/03_caracteres' 
# Carpeta para guardar ambos modelos
PROJECT_DIR = '../../models/03_caracteres'
os.makedirs(PROJECT_DIR, exist_ok=True)

# Parámetros Globales
IMG_SIZE = 64      # Tamaño de entrada (caracteres son pequeños)
BATCH_SIZE = 64
EPOCHS = 20        # Suficiente para caracteres simples
NUM_CLASSES = 36   # A-Z (26) + 0-9 (10) = 36 (Ajustar si tienes menos)

print("Entorno listo.")

PyTorch Version: 2.9.1+cu126
GPU Activa: NVIDIA GeForce RTX 4070 Ti
Entorno listo.


In [7]:
# Transformaciones para la CNN Custom (Normalización estándar)
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    # Normalización estándar para imágenes RGB
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Cargar Datasets usando ImageFolder (detecta carpetas A, B, C...)
train_dir = os.path.join(DATASET_DIR, 'train')
val_dir = os.path.join(DATASET_DIR, 'val')

# Verificación de seguridad
if not os.path.exists(train_dir):
    raise FileNotFoundError(f"No se encuentra el dataset en {train_dir}")

train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Guardar nombres de clases
CLASS_NAMES = train_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

print(f"Datos cargados:")
print(f"   - Clases ({NUM_CLASSES}): {CLASS_NAMES[:5]}...")
print(f"   - Train images: {len(train_dataset)}")
print(f"   - Val images:   {len(val_dataset)}")

Datos cargados:
   - Clases (36): ['0', '1', '2', '3', '4']...
   - Train images: 168588
   - Val images:   42198


In [11]:
print("\ROUND 1: Entrenando YOLOv8 Nano (Classifier)...")

# Nombre del experimento YOLO
run_name_yolo = f"{datetime.now().strftime('%Y%m%d')}_ocr_yolo_v8n"

# Cargar modelo
model_yolo = YOLO('yolov8n-cls.pt')

# Entrenar
start_time = time.time()
results_yolo = model_yolo.train(
    data=DATASET_DIR,
    project=PROJECT_DIR,
    name=run_name_yolo,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    pretrained=True,
    device=0 if torch.cuda.is_available() else 'cpu',
    verbose=True
)
yolo_train_time = time.time() - start_time

print(f"YOLO entrenamiento completado en {yolo_train_time:.2f} segundos.")

\ROUND 1: Entrenando YOLOv8 Nano (Classifier)...
New https://pypi.org/project/ultralytics/8.3.234 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/03_caracteres, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=64, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosai

In [12]:
class CustomOCR_CNN(nn.Module):
    def __init__(self, num_classes):
        super(CustomOCR_CNN, self).__init__()
        
        # Bloque Convolucional 1
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # Salida: 32x32
        )
        
        # Bloque Convolucional 2
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # Salida: 16x16
        )
        
        # Bloque Convolucional 3
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # Salida: 8x8
        )
        
        # Clasificador (Fully Connected)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 512), # Ajustar según IMG_SIZE (64 -> 8x8 feature map)
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.fc(x)
        return x

print("Arquitectura CNN definida.")

Arquitectura CNN definida.


In [13]:
print("\nROUND 2: Entrenando CNN Personalizada...")

# Instanciar modelo
cnn_model = CustomOCR_CNN(num_classes=NUM_CLASSES).to(DEVICE)

# Configuración
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

# Historial
cnn_history = {'loss': [], 'acc': []}

start_time = time.time()

for epoch in range(EPOCHS):
    cnn_model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # Barra de progreso
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
    
    for inputs, labels in pbar:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        # Forward
        outputs = cnn_model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward
        loss.backward()
        optimizer.step()
        
        # Métricas
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total
    cnn_history['loss'].append(epoch_loss)
    cnn_history['acc'].append(epoch_acc)
    
    print(f"   Epoch {epoch+1}: Loss={epoch_loss:.4f}, Accuracy={epoch_acc:.4f}")

cnn_train_time = time.time() - start_time
print(f"CNN Entrenamiento completado en {cnn_train_time:.2f} segundos.")

# Guardar modelo CNN
cnn_save_path = os.path.join(PROJECT_DIR, 'custom_cnn_best.pth')
torch.save(cnn_model.state_dict(), cnn_save_path)
print(f"Modelo guardado en: {cnn_save_path}")


ROUND 2: Entrenando CNN Personalizada...


Epoch 1/50:   0%|          | 0/86 [00:00<?, ?it/s]

   Epoch 1: Loss=3.7714, Accuracy=0.0505


Epoch 2/50:   0%|          | 0/86 [00:00<?, ?it/s]

   Epoch 2: Loss=3.4054, Accuracy=0.0747


...

   Epoch 48: Loss=1.5777, Accuracy=0.4460


Epoch 49/50:   0%|          | 0/86 [00:00<?, ?it/s]

   Epoch 49: Loss=1.5892, Accuracy=0.4475


Epoch 50/50:   0%|          | 0/86 [00:00<?, ?it/s]

   Epoch 50: Loss=1.5660, Accuracy=0.4588
CNN Entrenamiento completado en 89.90 segundos.
Modelo guardado en: ../../models/03_caracteres/custom_cnn_best.pth


In [8]:
def evaluate_model(model, loader, model_type='cnn'):
    model.eval() # Modo evaluación (importante para Dropout/BatchNorm)
    all_preds = []
    all_labels = []
    inference_times = []
    
    print(f"Evaluando {model_type}...")
    
    with torch.no_grad():
        for inputs, labels in tqdm(loader):
            # Preparar inputs según modelo
            if model_type == 'cnn':
                inputs = inputs.to(DEVICE)
            else:
                # YOLO espera rutas de archivos o arrays numpy, pero aquí
                # estamos pasando tensores. Para ser justos en velocidad,
                # usaremos el método .predict() de YOLO en lote si es posible,
                # o convertiremos. Para simplificar, usaremos validación integrada de YOLO arriba
                # y aquí solo mediremos CNN, luego uniremos datos.
                pass 

            if model_type == 'cnn':
                start = time.time()
                outputs = model(inputs)
                end = time.time()
                
                _, predicted = outputs.max(1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.numpy())
                
                # Tiempo por batch -> Tiempo por imagen
                inference_times.append((end - start) / inputs.size(0))

    if model_type == 'cnn':
        acc = accuracy_score(all_labels, all_preds)
        avg_inference_ms = (sum(inference_times) / len(inference_times)) * 1000
        return acc, avg_inference_ms

# 1. Evaluar CNN
cnn_acc, cnn_speed = evaluate_model(cnn_model, val_loader, 'cnn')

# 2. Obtener métricas de YOLO (del objeto results)
# YOLO calcula esto automáticamente al entrenar
yolo_metrics = model_yolo.val()
yolo_acc = yolo_metrics.top1
yolo_speed = yolo_metrics.speed['inference'] # ms per image

# --- REPORTE FINAL ---
comparison_data = {
    'Modelo': ['YOLOv8n-CLS', 'Custom CNN'],
    'Accuracy (Top-1)': [yolo_acc, cnn_acc],
    'Tiempo Entrenamiento (s)': [yolo_train_time, cnn_train_time],
    'Velocidad Inferencia (ms/img)': [yolo_speed, cnn_speed]
}

df_results = pd.DataFrame(comparison_data)

print("\n🏆 TABLA DE RESULTADOS:")
display(df_results)

# Graficar
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

# Gráfica 1: Accuracy
sns.barplot(data=df_results, x='Modelo', y='Accuracy (Top-1)', ax=ax[0], palette='viridis')
ax[0].set_title('Precisión (Más alto es mejor)')
ax[0].set_ylim(0, 1.1)
for i, v in enumerate(df_results['Accuracy (Top-1)']):
    ax[0].text(i, v + 0.02, f"{v:.2%}", ha='center')

# Gráfica 2: Velocidad
sns.barplot(data=df_results, x='Modelo', y='Velocidad Inferencia (ms/img)', ax=ax[1], palette='rocket')
ax[1].set_title('Latencia (Más bajo es mejor)')
for i, v in enumerate(df_results['Velocidad Inferencia (ms/img)']):
    ax[1].text(i, v + 0.1, f"{v:.2f} ms", ha='center')

# Gráfica 3: Tiempo Entrenamiento
sns.barplot(data=df_results, x='Modelo', y='Tiempo Entrenamiento (s)', ax=ax[2], palette='magma')
ax[2].set_title('Costo de Entrenamiento (Segundos)')

plt.tight_layout()
plt.show()

Evaluando cnn...


  0%|          | 0/22 [00:00<?, ?it/s]

Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,480,996 parameters, 0 gradients, 3.3 GFLOPs
train: /home/roberto/moca_proyecto/datasets/03_caracteres/train... found 5504 images in 36 classes ✅ 
val: /home/roberto/moca_proyecto/datasets/03_caracteres/val... found 1373 images in 36 classes ✅ 
test: None...
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 138.9±42.7 MB/s, size: 1.8 KB)
val: Scanning /home/roberto/moca_proyecto/datasets/03_caracteres/val... 1373 images, 0 corrupt: 100% ━━━━━━━━━━━━ 1373/1373 4.8Mit/s 0.0s0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 86/86 300.6it/s 0.3s0.2s
                   all        0.7      0.943
Speed: 0.0ms preprocess, 0.2ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /home/roberto/moca_proyecto/notebooks/03_caracteres/runs/classify/val3

🏆 TABLA DE RESULTADOS:


,Modelo,Accuracy (Top-1),Tiempo Entrenamiento (s),Velocidad Inferencia (ms/img)
0,YOLOv8n-CLS,0.699927,31.136698,0.180391
1,Custom CNN,0.612527,27.832779,0.015901


/tmp/ipykernel_4787/1340333088.py:65: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df_results, x='Modelo', y='Accuracy (Top-1)', ax=ax[0], palette='viridis')
/tmp/ipykernel_4787/1340333088.py:72: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df_results, x='Modelo', y='Velocidad Inferencia (ms/img)', ax=ax[1], palette='rocket')
/tmp/ipykernel_4787/1340333088.py:78: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df_results, x='Modelo', y='Tiempo Entrenamiento (s)', ax=ax[2], palette='magma')


<Figure size 1800x500 with 3 Axes>

In [8]:
print("\ROUND 1: Entrenando YOLOv8 Nano (Classifier) para 30,000 imágenes...")

# Nombre del experimento YOLO
run_name_yolo = f"{datetime.now().strftime('%Y%m%d')}_ocr_yolo_v8n_30k"

# Cargar modelo
model_yolo = YOLO('yolov8n-cls.pt')

# Entrenar
start_time = time.time()
results_yolo = model_yolo.train(
    data=DATASET_DIR,
    project=PROJECT_DIR,
    name=run_name_yolo,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    pretrained=True,
    device=0 if torch.cuda.is_available() else 'cpu',
    verbose=True
)
yolo_train_time = time.time() - start_time

print(f"YOLO entrenamiento completado en {yolo_train_time:.2f} segundos.")

\ROUND 1: Entrenando YOLOv8 Nano (Classifier) para 30,000 imágenes...
New https://pypi.org/project/ultralytics/8.3.234 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/03_caracteres, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=64, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, 

In [9]:
class CustomOCR_CNN(nn.Module):
    def __init__(self, num_classes):
        super(CustomOCR_CNN, self).__init__()
        
        # Bloque Convolucional 1
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # Salida: 32x32
        )
        
        # Bloque Convolucional 2
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # Salida: 16x16
        )
        
        # Bloque Convolucional 3
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # Salida: 8x8
        )
        
        # Clasificador (Fully Connected)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 512), # Ajustar según IMG_SIZE (64 -> 8x8 feature map)
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.fc(x)
        return x

print("Arquitectura CNN definida.")

Arquitectura CNN definida.


In [10]:
print("\nROUND 2: Entrenando CNN Personalizada...")

# Instanciar modelo
cnn_model = CustomOCR_CNN(num_classes=NUM_CLASSES).to(DEVICE)

# Configuración
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

# Historial
cnn_history = {'loss': [], 'acc': []}

start_time = time.time()

for epoch in range(EPOCHS):
    cnn_model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # Barra de progreso
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
    
    for inputs, labels in pbar:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        # Forward
        outputs = cnn_model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward
        loss.backward()
        optimizer.step()
        
        # Métricas
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total
    cnn_history['loss'].append(epoch_loss)
    cnn_history['acc'].append(epoch_acc)
    
    print(f"   Epoch {epoch+1}: Loss={epoch_loss:.4f}, Accuracy={epoch_acc:.4f}")

cnn_train_time = time.time() - start_time
print(f"CNN Entrenamiento completado en {cnn_train_time:.2f} segundos.")

# Guardar modelo CNN
cnn_save_path = os.path.join(PROJECT_DIR, 'custom_cnn_best.pth')
torch.save(cnn_model.state_dict(), cnn_save_path)
print(f"Modelo guardado en: {cnn_save_path}")


ROUND 2: Entrenando CNN Personalizada...


Epoch 1/20:   0%|          | 0/2635 [00:00<?, ?it/s]

   Epoch 1: Loss=2.5670, Accuracy=0.2187


Epoch 2/20:   0%|          | 0/2635 [00:00<?, ?it/s]

   Epoch 2: Loss=2.2446, Accuracy=0.2844


...

   Epoch 18: Loss=0.5979, Accuracy=0.7959


Epoch 19/20:   0%|          | 0/2635 [00:00<?, ?it/s]

   Epoch 19: Loss=0.5694, Accuracy=0.8055


Epoch 20/20:   0%|          | 0/2635 [00:00<?, ?it/s]

   Epoch 20: Loss=0.5484, Accuracy=0.8130
CNN Entrenamiento completado en 1159.69 segundos.
Modelo guardado en: ../../models/03_caracteres/custom_cnn_best.pth


In [11]:
def evaluate_model(model, loader, model_type='cnn'):
    model.eval() # Modo evaluación (importante para Dropout/BatchNorm)
    all_preds = []
    all_labels = []
    inference_times = []
    
    print(f"Evaluando {model_type}...")
    
    with torch.no_grad():
        for inputs, labels in tqdm(loader):
            # Preparar inputs según modelo
            if model_type == 'cnn':
                inputs = inputs.to(DEVICE)
            else:
                # YOLO espera rutas de archivos o arrays numpy, pero aquí
                # estamos pasando tensores. Para ser justos en velocidad,
                # usaremos el método .predict() de YOLO en lote si es posible,
                # o convertiremos. Para simplificar, usaremos validación integrada de YOLO arriba
                # y aquí solo mediremos CNN, luego uniremos datos.
                pass 

            if model_type == 'cnn':
                start = time.time()
                outputs = model(inputs)
                end = time.time()
                
                _, predicted = outputs.max(1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.numpy())
                
                # Tiempo por batch -> Tiempo por imagen
                inference_times.append((end - start) / inputs.size(0))

    if model_type == 'cnn':
        acc = accuracy_score(all_labels, all_preds)
        avg_inference_ms = (sum(inference_times) / len(inference_times)) * 1000
        return acc, avg_inference_ms

# 1. Evaluar CNN
cnn_acc, cnn_speed = evaluate_model(cnn_model, val_loader, 'cnn')

# 2. Obtener métricas de YOLO (del objeto results)
# YOLO calcula esto automáticamente al entrenar
yolo_metrics = model_yolo.val()
yolo_acc = yolo_metrics.top1
yolo_speed = yolo_metrics.speed['inference'] # ms per image

# --- REPORTE FINAL ---
comparison_data = {
    'Modelo': ['YOLOv8n-CLS', 'Custom CNN'],
    'Accuracy (Top-1)': [yolo_acc, cnn_acc],
    'Tiempo Entrenamiento (s)': [yolo_train_time, cnn_train_time],
    'Velocidad Inferencia (ms/img)': [yolo_speed, cnn_speed]
}

df_results = pd.DataFrame(comparison_data)

print("\nTABLA DE RESULTADOS:")
display(df_results)

# Graficar
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

# Gráfica 1: Accuracy
sns.barplot(data=df_results, x='Modelo', y='Accuracy (Top-1)', ax=ax[0], palette='viridis')
ax[0].set_title('Precisión (Más alto es mejor)')
ax[0].set_ylim(0, 1.1)
for i, v in enumerate(df_results['Accuracy (Top-1)']):
    ax[0].text(i, v + 0.02, f"{v:.2%}", ha='center')

# Gráfica 2: Velocidad
sns.barplot(data=df_results, x='Modelo', y='Velocidad Inferencia (ms/img)', ax=ax[1], palette='rocket')
ax[1].set_title('Latencia (Más bajo es mejor)')
for i, v in enumerate(df_results['Velocidad Inferencia (ms/img)']):
    ax[1].text(i, v + 0.1, f"{v:.2f} ms", ha='center')

# Gráfica 3: Tiempo Entrenamiento
sns.barplot(data=df_results, x='Modelo', y='Tiempo Entrenamiento (s)', ax=ax[2], palette='magma')
ax[2].set_title('Costo de Entrenamiento (Segundos)')

plt.tight_layout()
plt.show()

Evaluando cnn...


  0%|          | 0/660 [00:00<?, ?it/s]

Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,480,996 parameters, 0 gradients, 3.3 GFLOPs
train: /home/roberto/moca_proyecto/datasets/03_caracteres/train... found 168588 images in 36 classes ✅ 
val: /home/roberto/moca_proyecto/datasets/03_caracteres/val... found 42198 images in 36 classes ✅ 
test: None...
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 294.5±126.7 MB/s, size: 3.7 KB)
val: Scanning /home/roberto/moca_proyecto/datasets/03_caracteres/val... 42197 images, 1 corrupt: 100% ━━━━━━━━━━━━ 42197/42197 108.0Mit/s 0.0s
val: /home/roberto/moca_proyecto/datasets/03_caracteres/val/1/R-418-KOQ_6087_2.jpg: ignoring corrupt image/label: image size (19, 9) <10 pixels
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2638/2638 399.0it/s 6.6s<0.0s
                   all       0.95      0.994
Speed: 0.0ms preprocess, 0.1ms inference, 0.0ms loss, 0.0ms postprocess per image
Result

,Modelo,Accuracy (Top-1),Tiempo Entrenamiento (s),Velocidad Inferencia (ms/img)
0,YOLOv8n-CLS,0.949949,921.477282,0.132728
1,Custom CNN,0.930755,1159.685794,0.007900


/tmp/ipykernel_9559/2504788383.py:65: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df_results, x='Modelo', y='Accuracy (Top-1)', ax=ax[0], palette='viridis')
/tmp/ipykernel_9559/2504788383.py:72: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df_results, x='Modelo', y='Velocidad Inferencia (ms/img)', ax=ax[1], palette='rocket')
/tmp/ipykernel_9559/2504788383.py:78: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df_results, x='Modelo', y='Tiempo Entrenamiento (s)', ax=ax[2], palette='magma')


<Figure size 1800x500 with 3 Axes>